# **INKREPAIR - DATASET GENERATOR**


# **DATA**

In [ ]:
# CODE BLOCK 1 (IMPORT DATASET)

import os
import shutil

datasetpath = "/content/dataset"

if os.path.exists(datasetpath):
    shutil.rmtree(datasetpath)
    print(f"Deleted '{datasetpath}'")
elif not os.path.exists(datasetpath):
    print(f"'{datasetpath}' does not exist, creating...")
    os.makedirs(datasetpath)

import gdown
output = "/content/dataset.zip"
gdown.download("https://drive.google.com/file/d/1epGFmhJwfeZaKoM-TsqqH-b_VSl6kLMF/view?usp=drive_link", output, quiet=False, fuzzy=True)
!unzip /content/dataset.zip -d {datasetpath}

# ***LINE DELETION***

In [ ]:
# CODE BLOCK 2a (JUNCTION STUB DELETION)
import random
import numpy as np
from PIL import Image
from scipy import ndimage as ndi
from skimage.morphology import skeletonize
# Uses _K8 and _grow from CODE BLOCK 2b

_OFFS = [(dy, dx) for dy in (-1, 0, 1) for dx in (-1, 0, 1) if dy or dx]
_TRIP = [(0, 1, 2), (0, 2, 1), (1, 2, 0)]          # (bar a, bar b, stem)


class _UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        self.p[self.find(a)] = self.find(b)


def _group_coords(lab, n):
    """Pixel coords sorted by label: coords[start[k]:start[k+1]] are label k's pixels."""
    m = lab > 0
    coords = np.argwhere(m)
    l = lab[m]
    order = np.argsort(l, kind="stable")
    return coords[order], np.searchsorted(l[order], np.arange(n + 2))


def _branch_graph(skel):
    """Junction nodes, branches, and arms = (branch, node, end pixel) for every
    branch end that touches a node."""
    nb = ndi.convolve(skel.astype(np.int32), _K8, mode="constant") - skel
    core = skel & (nb >= 3)
    node = skel & ndi.binary_dilation(core, structure=_K8)
    branches = skel & ~node
    nlab, nnode = ndi.label(node, structure=_K8)
    blab, nbr = ndi.label(branches, structure=_K8)
    blen = np.bincount(blab.ravel(), minlength=nbr + 1)

    bnb = ndi.convolve(branches.astype(np.int32), _K8, mode="constant") - branches
    ey, ex = np.nonzero(branches & (bnb <= 1))
    nl_p = np.pad(nlab, 1)
    arms, seen = [], set()
    for dy, dx in _OFFS:
        v = nl_p[ey + 1 + dy, ex + 1 + dx]
        k = v > 0
        for y, x, n in zip(ey[k], ex[k], v[k]):
            key = (int(y), int(x), int(n))
            if key not in seen:
                seen.add(key)
                arms.append((int(blab[y, x]), int(n), (int(y), int(x))))
    return nlab, nnode, core, blab, nbr, blen, arms


class JunctionStubDeletion:
    """
    At T-like junctions, erases the start of the line that arrives (the stem)
    while keeping the line it touches (the bar) intact.
    The number of stubs is a fraction of the eligible junctions in the image.
    """
    def __init__(
        self,
        stub_frac=(0.3, 0.7),     # fraction of eligible T-junctions cut, drawn per image
        max_stubs=None,           # optional hard cap per image
        stub_length=(4, 40),      # VISIBLE removed length, measured from the bar's edge
        direction_len=15,         # px along each branch used to estimate its direction
        through_min_angle=145,    # bar must be at least this straight (degrees)
        angle_gap=20,             # bar pair must be this much straighter than the next pair
        min_stem_len=40,          # shorter stems are never cut
        max_stem_frac=0.5,        # max fraction of a stem that can be removed
        one_stub_per_stem=True,   # False: a stem may be cut at both ends (floating pieces)
        min_component_len=120,    # shapes with less skeleton than this are protected
        artifact_len=None,        # spur / split-junction length; None = auto (~ stroke width)
        prune_iters=2,            # spur pruning passes
        threshold=180,            # ink = gray < threshold (for the skeleton)
        erase_threshold=250,      # also erase anti-aliased fringe up to this gray level
        radius_pad=0.5,           # extra px of protection around the bar
    ):
        self.stub_frac = stub_frac
        self.max_stubs = max_stubs
        self.stub_length = stub_length
        self.direction_len = direction_len
        self.through_min_angle = through_min_angle
        self.angle_gap = angle_gap
        self.min_stem_len = min_stem_len
        self.max_stem_frac = max_stem_frac
        self.one_stub_per_stem = one_stub_per_stem
        self.min_component_len = min_component_len
        self.artifact_len = artifact_len
        self.prune_iters = prune_iters
        self.threshold = threshold
        self.erase_threshold = erase_threshold
        self.radius_pad = radius_pad
        self.last_stats = {}

    def __call__(self, img):
        if not isinstance(img, Image.Image):
            img = Image.fromarray((img.squeeze().numpy() * 255).astype(np.uint8))
        gray = np.asarray(img.convert("L"))
        rng = np.random.default_rng(random.getrandbits(64))   # respects random.seed()
        self.last_stats = {}

        skel = skeletonize(gray < self.threshold)
        if not skel.any():
            return img
        loose = gray < self.erase_threshold
        rad = ndi.distance_transform_edt(loose)       # local stroke half-width (incl. fringe)
        skel, stub = self._place_stubs(skel, rad, rng) # skel comes back pruned
        if not stub.any():
            return img

        # Voronoi assignment: each pixel belongs to its nearest skeleton pixel
        _, (iy, ix) = ndi.distance_transform_edt(~skel, return_indices=True)

        # Protect every kept stroke: union of disks along the kept skeleton
        keep = skel & ~stub
        if keep.any():
            d, (ky, kx) = ndi.distance_transform_edt(~keep, return_indices=True)
            protected = d <= rad[ky, kx] + self.radius_pad
        else:
            protected = np.zeros_like(skel)

        erase = stub[iy, ix] & loose & ~protected
        if not erase.any():
            return img
        out = gray.copy()
        out[erase] = 255
        out_img = Image.fromarray(out)
        return out_img.convert(img.mode) if img.mode != "L" else out_img

    def _place_stubs(self, skel, rad, rng):
        st = dict(junctions=0, three_way=0, ambiguous=0, short_stem=0,
                  small_shape=0, eligible=0, target=0, placed=0, artifact_len=0)
        self.last_stats = st
        stub = np.zeros_like(skel)

        r_med = float(np.median(rad[skel]))
        art = (self.artifact_len if self.artifact_len is not None
               else int(np.ceil(2 * r_med)) + 2)
        st["artifact_len"] = art

        # 1. Prune short spurs (skeletonization noise that turns T's into 4-way nodes)
        for it in range(self.prune_iters + 1):
            nlab, nnode, core, blab, nbr, blen, arms = _branch_graph(skel)
            if it == self.prune_iters or nnode == 0:
                break
            n_arms = np.bincount(np.array([a[0] for a in arms], dtype=np.int64),
                                 minlength=nbr + 1)
            spur = (n_arms == 1) & (blen < art)
            spur[0] = False
            if not spur.any():
                break
            skel = skeletonize(skel & ~spur[blab])
        if nnode == 0 or nbr == 0:
            return skel, stub

        # 2. Merge nodes joined by very short branches into one junction cluster
        uf = _UnionFind(nnode + 1)
        b_nodes = {}
        for b, n, _ in arms:
            b_nodes.setdefault(b, []).append(n)
        internal = np.zeros(nbr + 1, bool)
        for b, ns in b_nodes.items():
            if len(ns) >= 2 and blen[b] <= art:
                internal[b] = True
                for n in ns[1:]:
                    uf.union(ns[0], n)
        roots = np.array([uf.find(n) for n in range(nnode + 1)])

        cl_arms = {}
        for b, n, p in arms:
            if not internal[b]:
                cl_arms.setdefault(int(roots[n]), []).append((b, p))
        st["junctions"] = len(cl_arms)

        members = {}
        for n in range(1, nnode + 1):
            members.setdefault(int(roots[n]), []).append(n)
        cl_internal = {}
        for b in np.flatnonzero(internal):
            cl_internal.setdefault(int(roots[b_nodes[int(b)][0]]), []).append(int(b))

        # Cluster centres = core-pixel centroid of all member nodes
        idx = np.arange(1, nnode + 1)
        cnt = np.asarray(ndi.sum(core, nlab, idx), float)
        com = np.array(ndi.center_of_mass(core, nlab, idx), float).reshape(-1, 2)
        wsum = np.bincount(roots[1:], weights=cnt, minlength=nnode + 1)
        ccy = np.bincount(roots[1:], weights=cnt * com[:, 0], minlength=nnode + 1) / np.maximum(wsum, 1e-9)
        ccx = np.bincount(roots[1:], weights=cnt * com[:, 1], minlength=nnode + 1) / np.maximum(wsum, 1e-9)

        clab, _ = ndi.label(skel, structure=_K8)
        comp_len = np.bincount(clab.ravel())

        # 3. Find every eligible T-junction
        eligible = []
        for root, al in cl_arms.items():
            if len(al) != 3:
                continue
            st["three_way"] += 1
            cy, cx = ccy[root], ccx[root]
            dirs, zones = [], []
            for b, p in al:
                z = _grow(blab, b, p, self.direction_len)
                fy, fx = max(z, key=z.get)
                v = np.array([fy - cy, fx - cx], dtype=float)
                dirs.append(v / (np.linalg.norm(v) + 1e-9))
                zones.append(z)
            ang = [np.degrees(np.arccos(np.clip(dirs[a] @ dirs[b], -1, 1)))
                   for a, b, _ in _TRIP]
            order = np.argsort(ang)[::-1]
            if (ang[order[0]] < self.through_min_angle
                    or ang[order[0]] - ang[order[1]] < self.angle_gap):
                st["ambiguous"] += 1
                continue
            a, b2, s = _TRIP[order[0]]
            stem, sp = al[s]
            if stem in (al[a][0], al[b2][0]):       # stem loops back into the bar
                st["ambiguous"] += 1
                continue
            if blen[stem] < self.min_stem_len:
                st["short_stem"] += 1
                continue
            if comp_len[clab[sp]] < self.min_component_len:
                st["small_shape"] += 1
                continue
            bar_px = np.array(list(zones[a]) + list(zones[b2])).T
            eligible.append((root, stem, sp, bar_px))
        st["eligible"] = len(eligible)
        if not eligible:
            return skel, stub

        # 4. Relative budget
        target = int(round(rng.uniform(*self.stub_frac) * len(eligible)))
        if self.max_stubs is not None:
            target = min(target, self.max_stubs)
        st["target"] = target

        ncoords, nstart = _group_coords(nlab, nnode)
        bcoords, bstart = _group_coords(blab, nbr)
        lo, hi = self.stub_length
        removed = np.zeros(nbr + 1, np.int64)
        stubbed = np.zeros(nbr + 1, bool)
        placed = 0

        for i in rng.permutation(len(eligible)):
            if placed >= target:
                break
            root, stem, sp, bar_px = eligible[i]
            if self.one_stub_per_stem and stubbed[stem]:
                continue
            r_bar = float(np.median(rad[bar_px[0], bar_px[1]]))
            offset = int(np.ceil(r_bar))                 # part hidden under the bar
            g = int(rng.integers(lo, hi + 1)) + offset
            g = min(g, int(self.max_stem_frac * blen[stem]) - int(removed[stem]))
            if g - offset < lo:
                continue

            seg = _grow(blab, stem, sp, g - 1)
            sy, sx = np.array(list(seg)).T
            stub[sy, sx] = True

            # Junction ring pixels go to the cut side; core keeps the BAR's width
            px = np.concatenate([ncoords[nstart[n]:nstart[n + 1]] for n in members[root]])
            is_core = core[px[:, 0], px[:, 1]]
            stub[px[~is_core, 0], px[~is_core, 1]] = True
            rad[px[is_core, 0], px[is_core, 1]] = r_bar
            for b in cl_internal.get(root, []):
                q = bcoords[bstart[b]:bstart[b + 1]]
                rad[q[:, 0], q[:, 1]] = r_bar

            removed[stem] += len(seg)
            stubbed[stem] = True
            placed += 1

        st["placed"] = placed
        return skel, stub


In [ ]:
# CODE BLOCK 2b (SkeletonGapDeletion)
import random
from collections import deque
import numpy as np
from PIL import Image
from scipy import ndimage as ndi
from skimage.morphology import skeletonize

_K8 = np.ones((3, 3), dtype=np.int32)


def _grow(blab, b, start, radius):
    """BFS along branch b from start, up to `radius` steps -> {pixel: distance}."""
    H, W = blab.shape
    dist = {start: 0}
    q = deque([start])
    while q:
        p = q.popleft()
        d = dist[p]
        if d == radius:
            continue
        y, x = p
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                n = (y + dy, x + dx)
                if n in dist or not (0 <= n[0] < H and 0 <= n[1] < W):
                    continue
                if blab[n] != b:
                    continue
                dist[n] = d + 1
                q.append(n)
    return dist


class SkeletonGapDeletion:
    """
    Cuts gaps along the line skeleton, with budgets so no stroke or small
    shape (eyes, pupils, highlights) is ever removed entirely.
    """
    def __init__(
        self,
        num_deletions=(10, 30),     # holes per image (min, max); ignored if holes_per_kpx is set
        holes_per_kpx=None,         # optional: holes per 1000 skeleton px (relative budget)
        gap_length=(8, 60),         # hole length along the stroke, in px
        margin=8,                   # min distance to endpoints, junctions and other holes
        min_branch_len=30,          # shorter branches are never cut
        max_branch_frac=0.35,       # max fraction of one branch that can be removed
        min_component_len=120,      # shapes with less skeleton than this are protected
        max_component_frac=0.25,    # max fraction of one shape that can be removed
        threshold=180,              # ink = gray < threshold (for the skeleton)
        erase_threshold=250,        # also erase anti-aliased fringe up to this gray level
    ):
        self.num_deletions = num_deletions
        self.holes_per_kpx = holes_per_kpx
        self.gap_length = gap_length
        self.margin = margin
        self.min_branch_len = min_branch_len
        self.max_branch_frac = max_branch_frac
        self.min_component_len = min_component_len
        self.max_component_frac = max_component_frac
        self.threshold = threshold
        self.erase_threshold = erase_threshold

    def __call__(self, img):
        if not isinstance(img, Image.Image):
            img = Image.fromarray((img.squeeze().numpy() * 255).astype(np.uint8))
        gray = np.asarray(img.convert("L"))
        rng = np.random.default_rng(random.getrandbits(64))   # respects random.seed()

        skel = skeletonize(gray < self.threshold)
        if not skel.any():
            return img
        hole = self._place_holes(skel, rng)
        if not hole.any():
            return img

        # Assign each pixel to its nearest skeleton pixel -> clean full-width cut
        _, (iy, ix) = ndi.distance_transform_edt(~skel, return_indices=True)
        erase = hole[iy, ix] & (gray < self.erase_threshold)
        out = gray.copy()
        out[erase] = 255
        out_img = Image.fromarray(out)
        return out_img.convert(img.mode) if img.mode != "L" else out_img

    def _place_holes(self, skel, rng):
        hole = np.zeros_like(skel)

        # Split the skeleton into branches by removing junctions
        nb = ndi.convolve(skel.astype(np.int32), _K8, mode="constant") - skel
        junctions = skel & (nb >= 3)
        branches = skel & ~ndi.binary_dilation(junctions, structure=_K8)
        blab, nbr = ndi.label(branches, structure=_K8)
        if nbr == 0:
            return hole
        bnb = ndi.convolve(branches.astype(np.int32), _K8, mode="constant") - branches
        ends = branches & (bnb <= 1)          # branch endpoints (closed loops have none)

        # Connected shapes: protect small ones, cap removal per shape
        clab, _ = ndi.label(skel, structure=_K8)
        comp_len = np.bincount(clab.ravel())
        branch_len = np.bincount(blab.ravel(), minlength=nbr + 1)
        branch_comp = np.zeros(nbr + 1, dtype=np.int64)
        branch_comp[blab[branches]] = clab[branches]

        ok = ((branch_len >= self.min_branch_len)
              & (comp_len[branch_comp] >= self.min_component_len))
        ok[0] = False
        cand = np.flatnonzero(ok)
        if len(cand) == 0:
            return hole

        # Branch pixel coordinates grouped by label, for fast sampling
        coords = np.argwhere(branches)
        lab = blab[branches]
        order = np.argsort(lab, kind="stable")
        coords, lab = coords[order], lab[order]
        start = np.searchsorted(lab, np.arange(nbr + 2))

        w = branch_len[cand].astype(float)
        w /= w.sum()                           # longer branches get more holes

        if self.holes_per_kpx is not None:
            target = int(round(skel.sum() / 1000 * self.holes_per_kpx * rng.uniform(0.5, 1.5)))
        else:
            target = int(rng.integers(self.num_deletions[0], self.num_deletions[1] + 1))

        removed_b = np.zeros(nbr + 1, dtype=np.int64)
        removed_c = np.zeros_like(comp_len)
        placed = tries = 0
        while placed < target and tries < 30 * max(target, 1):
            tries += 1
            b = cand[rng.choice(len(cand), p=w)]
            c = branch_comp[b]
            g = int(rng.integers(self.gap_length[0], self.gap_length[1] + 1))
            if removed_b[b] + g > self.max_branch_frac * branch_len[b]:
                continue
            if removed_c[c] + g > self.max_component_frac * comp_len[c]:
                continue

            y, x = coords[rng.integers(start[b], start[b + 1])]
            r = g // 2
            reach = r + self.margin
            zone = _grow(blab, b, (int(y), int(x)), reach)
            # Branch too short around this point, or zone would swallow a whole loop
            if max(zone.values()) < reach or len(zone) >= branch_len[b]:
                continue
            zy, zx = np.array(list(zone)).T
            if ends[zy, zx].any() or hole[zy, zx].any():
                continue   # too close to an endpoint/junction or to another hole

            seg = np.array([p for p, d in zone.items() if d <= r]).T
            hole[seg[0], seg[1]] = True
            removed_b[b] += seg.shape[1]
            removed_c[c] += seg.shape[1]
            placed += 1

        return hole

In [ ]:
# CODE BLOCK 2c (general deletion)

class LineCorruption:
    """
    General corruption: junction stubs first, then skeleton gaps.
    last_masks holds the pixels erased by each stage (for visualization).
    """
    def __init__(self, junction_kwargs=None, gap_kwargs=None,
                 p_junction=1.0, p_gap=1.0):
        self.junction = JunctionStubDeletion(**(junction_kwargs or {}))
        self.gap = SkeletonGapDeletion(**(gap_kwargs or {}))
        self.p_junction = p_junction
        self.p_gap = p_gap
        self.last_masks = None

    def __call__(self, img):
        if not isinstance(img, Image.Image):
            img = Image.fromarray((img.squeeze().numpy() * 255).astype(np.uint8))
        out = img
        if random.random() < self.p_junction:
            out = self.junction(out)
        after_junction = out
        if random.random() < self.p_gap:
            out = self.gap(out)

        g0 = np.asarray(img.convert("L"))
        g1 = np.asarray(after_junction.convert("L"))
        g2 = np.asarray(out.convert("L"))
        self.last_masks = dict(junction=g1 > g0, gap=g2 > g1)
        return out          # same object as img if nothing was erased


# General line deletion parametrization
# Used by the check blocks (2e-2g) and copied into CORRUPTION_KWARGS in CODE BLOCK 3.
# All lengths are in pixels, at the training resolution (896 x 896).
CHECK_KWARGS = dict(

    junction_kwargs = dict(
      stub_frac=(0.5, 1.0),     # fraction of eligible T-junctions cut per image
      max_stubs=None,           # optional cap
      stub_length=(2, 20),      # visible length, from the bar's edge
      min_stem_len=20,
      max_stem_frac=0.5,
      min_component_len=120,
      threshold=180,
    ),

    # Skeleton gaps: cut holes in the middle of lines, away from ends and junctions.
    gap_kwargs = dict(
        num_deletions=(4, 12),    # target number of gaps per image, drawn uniformly per image.
        gap_length=(4, 60),       # length of each gap along the line, drawn uniformly per gap.
        margin=8,                 # minimum distance between a gap and any line end, junction,
                                  # or other gap, so each gap has intact line on both sides.
        min_branch_len=30,        # line segments (between junctions/ends) shorter than this
                                  # are never cut.
        max_branch_frac=0.35,     # at most this fraction of one line segment can be removed
                                  # in total, across all gaps on it.
        min_component_len=120,    # shapes whose total centreline is shorter than this are
                                  # never touched.
        max_component_frac=0.25,  # at most this fraction of one connected shape can be
                                  # removed in total, so no shape is erased entirely.
        threshold=180,            # gray level below which a pixel counts as ink.
        holes_per_kpx=None,       # if set (e.g. 3), the gap count scales with the amount
                                  # of line in the image instead of using num_deletions.
        erase_threshold=250,      # also erase the grey anti-aliased edge of the line.
    ),

    # Stage probabilities (per image): junction stubs run first, then gaps.
    p_junction = 1.0,             # probability that an image gets junction stubs (1.0 = always).
    p_gap      = 1.0,             # probability that an image gets gaps (1.0 = always).
                                  # Set both below 1 to get some easier or intact images.
)


In [ ]:
# CODE BLOCK 2d (CHECK UTILITIES)
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi
from skimage.morphology import skeletonize

CHECK_SIZE  = 896          # same as IMAGE_SIZE in CODE BLOCK 3
CHECK_N     = 10           # number of images to inspect
CHECK_SEED  = 0            # change to see other images / other holes
CHECK_FILES = None         # or a list of file names to inspect specific images
ZOOM        = 160          # size of each zoom crop, in px
N_ZOOMS     = 2            # zoom panels per image

_S8 = np.ones((3, 3), dtype=bool)

clean_src = datasetpath
all_files = sorted(f for f in os.listdir(clean_src)
                   if f.lower().endswith((".png", ".jpg", ".jpeg")))
check_files = CHECK_FILES or random.Random(CHECK_SEED).sample(all_files, min(CHECK_N, len(all_files)))

# Loaded once, shared by the three checks (resized like training)
CHECK_IMAGES = [(f, Image.open(os.path.join(clean_src, f)).convert("L")
                     .resize((CHECK_SIZE, CHECK_SIZE), Image.BILINEAR))
                for f in check_files]
print("Check images:", check_files)


# how to colour the erased pixels
def erased_mask(name, color):
    """Single-stage deleters: everything erased is one type."""
    def fn(deleter, g, c):
        return [(name, (c > 229.5) & (g < 127.5), color)]
    return fn

def stage_masks(deleter, g, c):
    """LineCorruption: split erased ink by stage using last_masks."""
    ink = g < 127.5
    return [("gaps",  deleter.last_masks["gap"] & ink,      (1.0, 0.0, 0.0)),
            ("stubs", deleter.last_masks["junction"] & ink, (1.0, 0.6, 0.0))]


# helpers
def _protected_shapes(ink, min_len):
    """Ink shapes whose skeleton is shorter than min_len (never cut)."""
    lab, _ = ndi.label(ink, structure=_S8)
    slab, _ = ndi.label(skeletonize(ink), structure=_S8)
    small = np.bincount(slab.ravel()) < min_len
    small[0] = False
    ids = np.unique(lab[small[slab]])
    ids = ids[ids > 0]
    return np.isin(lab, ids), len(ids)

def _zoom_centers(masks, n):
    """Centres of the largest holes, alternating between mask types."""
    per_type = []
    for name, m, _ in masks:
        hl, nh = ndi.label(m, structure=_S8)
        if not nh:
            continue
        sizes = np.bincount(hl.ravel())
        sizes[0] = 0
        lst = []
        for i in np.argsort(sizes)[::-1][:n]:
            if sizes[i] == 0:
                break
            ys, xs = np.nonzero(hl == i)
            lst.append((name, int(ys.mean()), int(xs.mean())))
        per_type.append(lst)
    out = []
    for i in range(n):
        for lst in per_type:
            if i < len(lst) and len(out) < n:
                out.append(lst[i])
    return out


def run_check(deleter, title, mask_fn, thr=180, min_len=120, n_zooms=N_ZOOMS):
    """Runs the deleter on CHECK_IMAGES and returns (rows, report) for display."""
    state = random.getstate()
    rows, report = [], []

    for k, (f, img) in enumerate(CHECK_IMAGES):
        random.seed(CHECK_SEED * 1000 + k)            # same holes every run
        t0  = time.perf_counter()
        cor = deleter(img)
        dt  = (time.perf_counter() - t0) * 1000

        g, c = np.asarray(img), np.asarray(cor)
        H, W = g.shape
        ink = g < thr
        erased = (c > 229.5) & (g < 127.5)            # same as LineLoss._hole_mask
        masks  = mask_fn(deleter, g, c)
        counts = {name: ndi.label(m, structure=_S8)[1] for name, m, _ in masks}

        # Per-shape damage: how much of each connected ink shape survived?
        lab, n_comp = ndi.label(ink, structure=_S8)
        idx = np.arange(1, n_comp + 1)
        total_px  = np.asarray(ndi.sum(ink, lab, idx))
        remain_px = np.asarray(ndi.sum(c < thr, lab, idx))
        vanished  = int((remain_px == 0).sum())
        heavy     = int(((1 - remain_px / np.maximum(total_px, 1)) > 0.5).sum())

        # Overlay: grey clean image, blue = protected shapes, coloured = erased
        protected, n_prot = _protected_shapes(ink, min_len)
        ov = np.repeat((g / 255.0)[..., None], 3, axis=2)
        ov[protected & ink] = (0.2, 0.4, 1.0)
        for _, m, col in masks:
            ov[m] = col

        # Zoom panels: corrupted | overlay
        zooms = []
        for name, cy, cx in _zoom_centers(masks, n_zooms):
            y0 = int(np.clip(cy - ZOOM // 2, 0, H - ZOOM))
            x0 = int(np.clip(cx - ZOOM // 2, 0, W - ZOOM))
            cc  = np.repeat((c[y0:y0 + ZOOM, x0:x0 + ZOOM] / 255.0)[..., None], 3, axis=2)
            oc  = ov[y0:y0 + ZOOM, x0:x0 + ZOOM]
            sep = np.full((ZOOM, 4, 3), 0.6)
            zooms.append((name, (y0, x0), np.concatenate([cc, sep, oc], axis=1)))

        share = erased.sum() / max((g < 127.5).sum(), 1)
        rows.append((f, c, ov, zooms, counts, share, vanished))
        report.append((f, counts, share, vanished, heavy, n_comp, n_prot, dt))

    random.setstate(state)
    return rows, report

In [ ]:
# CODE BLOCK 2d-2 (CHECK DISPLAY)
import numpy as np
import matplotlib.pyplot as plt

def show_check(rows, title, n_zooms=N_ZOOMS):
    """Figure: corrupted | overlay | zoom panels (corrupted | overlay), one row per image."""
    n = len(rows)
    fig, axes = plt.subplots(n, 2 + n_zooms, figsize=(5 * (2 + 2 * n_zooms), 5 * n),
                             gridspec_kw=dict(width_ratios=[1, 1] + [2] * n_zooms))
    axes = np.atleast_2d(axes)

    for r, (f, c, ov, zooms, counts, share, van) in enumerate(rows):
        cnt = ", ".join(f"{v} {k}" for k, v in counts.items())

        # Column 0: corrupted image
        axes[r, 0].imshow(c, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        axes[r, 0].set_title(f"{f} - corrupted", fontsize=10)
        axes[r, 0].axis("off")

        # Column 1: overlay (grey clean, blue protected, coloured erased)
        axes[r, 1].imshow(ov, interpolation="nearest")
        axes[r, 1].set_title(f"{cnt} | {share:.1%} of ink erased | vanished {van}",
                             fontsize=10, color="red" if van else "black")
        axes[r, 1].axis("off")

        # Remaining columns: zooms, with their location marked on the overlay
        for z in range(n_zooms):
            ax = axes[r, 2 + z]
            if z < len(zooms):
                name, (y0, x0), panel = zooms[z]
                ax.imshow(panel, interpolation="nearest")
                ax.set_title(f"zoom {z + 1} ({name}): corrupted | overlay", fontsize=10)
                axes[r, 1].add_patch(plt.Rectangle((x0, y0), ZOOM, ZOOM,
                                                   fill=False, ec="lime", lw=1.5))
                axes[r, 1].text(x0 + 3, y0 + 14, str(z + 1), color="lime", fontsize=10)
            ax.axis("off")

    fig.suptitle(f"{title}   (blue = protected shapes)", fontsize=14)
    plt.tight_layout()
    plt.show()


def print_check_summary(report, title):
    """Per-image table plus warnings."""
    keys = list(report[0][1].keys()) if report else []
    print(f"\n{title}")
    print(f"{'file':30s} " + " ".join(f"{k:>6s}" for k in keys)
          + f" {'erased/ink':>10s} {'vanished':>8s} {'>50%lost':>8s}"
          + f" {'shapes':>6s} {'protected':>9s} {'ms':>7s}")
    for f, counts, share, van, hv, nc, npr, dt in report:
        print(f"{f[:30]:30s} " + " ".join(f"{counts[k]:6d}" for k in keys)
              + f" {share:10.1%} {van:8d} {hv:8d} {nc:6d} {npr:9d} {dt:7.1f}")

    tot_vanished = sum(r[3] for r in report)
    mean_ms = np.mean([r[7] for r in report])
    print(f"\nmean corruption time: {mean_ms:.0f} ms/image")
    if tot_vanished:
        print(f"WARNING: {tot_vanished} shape(s) vanished entirely -> raise "
              f"min_component_len or lower the removal fractions")
    else:
        print("OK: no shape was erased entirely")
    if mean_ms > 150:
        print("NOTE: corruption may bottleneck the DataLoader (4 workers)")

In [ ]:
# CODE BLOCK 2e (CHECK JUNCTION STUBS ONLY)
jk = CHECK_KWARGS["junction_kwargs"]
title = "Junction stubs only"
rows, report = run_check(
    JunctionStubDeletion(**jk), title,
    mask_fn=erased_mask("stubs", (1.0, 0.6, 0.0)),        # orange
    thr=jk.get("threshold", 180),
    min_len=jk.get("min_component_len", 120),
    n_zooms=3,
)
show_check(rows, title, n_zooms=3)
print_check_summary(report, title)

# Where candidate junctions are lost
jd = JunctionStubDeletion(**jk)
for f, img in CHECK_IMAGES:
    random.seed(0)
    jd(img)
    print(f"{f[:28]:28s}", jd.last_stats)

In [ ]:
# CODE BLOCK 2f (CHECK SKELETON GAPS ONLY)
gk = CHECK_KWARGS["gap_kwargs"]
title = "Skeleton gaps only"
rows, report = run_check(
    SkeletonGapDeletion(**gk), title,
    mask_fn=erased_mask("gaps", (1.0, 0.0, 0.0)),         # red
    thr=gk.get("threshold", 180),
    min_len=gk.get("min_component_len", 120),
)
show_check(rows, title)
print_check_summary(report, title)

In [ ]:
# CODE BLOCK 2g (CHECK FULL LINE CORRUPTION)
gk = CHECK_KWARGS["gap_kwargs"]
title = "Full corruption (stubs + gaps)"
rows, report = run_check(
    LineCorruption(**CHECK_KWARGS), title,
    mask_fn=stage_masks,                                   # red = gaps, orange = stubs
    thr=gk.get("threshold", 180),
    min_len=gk.get("min_component_len", 120),
)
show_check(rows, title)
print_check_summary(report, title)

# ***TRAIN/TEST SPLIT***

In [ ]:
# CODE BLOCK 3 (TRAIN / TEST SPLIT)
# Builds: clean_training/ (90%), clean_test/ (10%, resized), corrupted_test/ (paired)
# Test pairs are generated ONCE and frozen on disk.

import os, json, random, shutil, copy, time
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from tqdm.auto import tqdm

SEED         = 1234
IMAGE_SIZE   = 896
TEST_FRAC    = 0.10
SAMPLE_N     = 4
USE_SYMLINKS = True          # training folder: symlink instead of copying ~9k files

# Corruption settings for train / test / sample, validated in CODE BLOCKS 2e-2g
# (CHECK_KWARGS, defined in 2c). Rerun this block when they change.
CORRUPTION_KWARGS = copy.deepcopy(CHECK_KWARGS)

clean_src = datasetpath
split_root     = "/content/split"
train_dir      = os.path.join(split_root, "clean_training")
test_clean_dir = os.path.join(split_root, "clean_test")
test_corr_dir  = os.path.join(split_root, "corrupted_test")

for d in (train_dir, test_clean_dir, test_corr_dir):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

# deterministic split
files = sorted(f for f in os.listdir(clean_src)
               if f.lower().endswith((".png", ".jpg", ".jpeg")))
assert files, f"No images found in {clean_src}"

sample_files = files[:SAMPLE_N]              # fixed preview images

rng  = random.Random(SEED)
rest = files[SAMPLE_N:]                      # everything except the previews
rng.shuffle(rest)
n_test  = int(round(len(files) * TEST_FRAC))
n_extra = max(n_test - SAMPLE_N, 0)          # previews count toward the 10%
test_files  = sorted(sample_files + rest[:n_extra])
train_files = sorted(rest[n_extra:])

print(f"Total {len(files)} | train {len(train_files)} | test {len(test_files)}")
print("Sample files:", sample_files)

# training folder
for f in tqdm(train_files, desc="train symlinks", unit="img"):
    src = os.path.abspath(os.path.join(clean_src, f))
    dst = os.path.join(train_dir, f)
    os.symlink(src, dst) if USE_SYMLINKS else shutil.copy2(src, dst)

# frozen test pairs
resize = transforms.Resize(
    (IMAGE_SIZE, IMAGE_SIZE),
    interpolation=transforms.InterpolationMode.BILINEAR,
    antialias=True,
)
deleter = LineCorruption(**CORRUPTION_KWARGS)

stats, empty_files = [], []
state = random.getstate()
t0 = time.perf_counter()
bar = tqdm(test_files, desc="test pairs", unit="img")
for k, f in enumerate(bar):
    img = Image.open(os.path.join(clean_src, f)).convert("L")
    img = resize(img)                       # corrupt at the SAME scale as training
    random.seed(SEED + 10_000 + k)          # reproducible holes, per image
    cor = deleter(img)

    # The deleter returns the SAME object when nothing was erased.
    if cor is img:
        empty_files.append(f)
        cor = img.copy()

    g = np.asarray(img, np.float32) / 255.0
    c = np.asarray(cor, np.float32) / 255.0
    ink  = int((g < 0.5).sum())
    hole = int(((c > 0.9) & (g < 0.5)).sum())
    stats.append((ink / g.size, hole / max(ink, 1)))

    stem = os.path.splitext(f)[0]
    img.save(os.path.join(test_clean_dir, stem + ".png"))
    cor.save(os.path.join(test_corr_dir,  stem + ".png"))

    # live info next to the bar: running mean of erased ink share, empty count
    bar.set_postfix(erased=f"{np.mean([s[1] for s in stats]):.3f}",
                    empty=len(empty_files))
random.setstate(state)
dt = time.perf_counter() - t0
print(f"Test pairs done in {dt/60:.1f} min "
      f"({dt / max(len(test_files), 1):.2f} s/image)")

test_files   = [os.path.splitext(f)[0] + ".png" for f in test_files]
sample_files = [os.path.splitext(f)[0] + ".png" for f in sample_files]

# corruption difficulty report
s = np.array(stats)
report = dict(
    ink_frac_mean     = float(s[:, 0].mean()),
    erased_share_mean = float(s[:, 1].mean()),
    erased_share_p10  = float(np.percentile(s[:, 1], 10)),
    erased_share_p90  = float(np.percentile(s[:, 1], 90)),
    n_empty           = len(empty_files),
)
spread = report['erased_share_p90'] / max(report['erased_share_p10'], 1e-9)
print(f"\nink_frac       {report['ink_frac_mean']:.4f}")
print(f"erased/ink     {report['erased_share_mean']:.3f}  "
      f"(p10 {report['erased_share_p10']:.3f}  p90 {report['erased_share_p90']:.3f}  "
      f"spread {spread:.1f}x)")

jk, gk = CORRUPTION_KWARGS["junction_kwargs"], CORRUPTION_KWARGS["gap_kwargs"]
gap_budget = (f"{gk['holes_per_kpx']} per 1000 px of line" if gk.get("holes_per_kpx")
              else f"{gk['num_deletions']}")
print(f"cuts/image     gaps {gap_budget} + junction stubs {jk.get('stub_frac')} of eligible T's"
      f"  (p_gap {CORRUPTION_KWARGS['p_gap']}, p_junction {CORRUPTION_KWARGS['p_junction']})")
print(f"empty (nothing erased): {report['n_empty']}"
      + (f" -> {empty_files[:5]}" if empty_files else ""))
if spread > 2.5:
    print("  NOTE: difficulty varies a lot across drawings - expected with a "
          "fixed hole count; consider gap_kwargs['holes_per_kpx'] for a relative budget.")

# manifest + sanity checks
manifest = dict(seed=SEED, image_size=IMAGE_SIZE, test_frac=TEST_FRAC,
                corruption=CORRUPTION_KWARGS, corruption_report=report,
                empty_files=empty_files,
                train=train_files, test=test_files, sample=sample_files)
with open(os.path.join(split_root, "split.json"), "w") as fh:
    json.dump(manifest, fh, indent=2)

assert not (set(train_files) & set(test_files)), "Train/test overlap!"
assert set(sample_files) <= set(test_files),     "Sample must come from the test set"
assert len(os.listdir(test_clean_dir)) == len(os.listdir(test_corr_dir)) == len(test_files)
print("Split OK ->", split_root)

In [ ]:
# CODE BLOCK 3b (PRE-GENERATE CORRUPTED TRAINING PAIRS)
# Run after CODE BLOCK 3. Corrupts every training image N_VARIANTS times, once,
# in parallel on all CPU cores, and saves the pairs to disk.
# Resumable: already-saved variants are skipped, so after a Colab disconnect
# just rerun this cell. If CORRUPTION_KWARGS changed, the old pairs are deleted.
import os, json, time, random, shutil
from multiprocessing import get_context
from PIL import Image
import torchvision.transforms as transforms
from tqdm.auto import tqdm

N_VARIANTS = 6                       # corrupted versions per training image
PRE_SEED   = SEED + 50_000           # different from the test-set seeds
N_PROC     = os.cpu_count()          # one process per core

pre_root     = os.path.join(split_root, "precorrupted")
pre_clean    = os.path.join(pre_root, "clean")       # resized clean images (stem.png)
pre_corr     = os.path.join(pre_root, "corrupted")   # stem_v0.png ... stem_v{K-1}.png
pre_manifest = os.path.join(pre_root, "manifest.json")

# N_VARIANTS is not part of the check: raising it later just adds variants
pre_settings = json.loads(json.dumps(dict(
    seed=PRE_SEED, image_size=IMAGE_SIZE, corruption=CORRUPTION_KWARGS)))
if os.path.exists(pre_manifest):
    with open(pre_manifest) as fh:
        if json.load(fh).get("settings") != pre_settings:
            print("Corruption settings changed -> deleting old pre-corrupted pairs")
            shutil.rmtree(pre_root)
os.makedirs(pre_clean, exist_ok=True)
os.makedirs(pre_corr, exist_ok=True)
with open(pre_manifest, "w") as fh:
    json.dump(dict(settings=pre_settings, n_variants=N_VARIANTS, complete=False), fh, indent=2)


def _save_atomic(img, path):
    """Write to a temp file then rename, so an interrupted save never leaves a
    half-written PNG that would be skipped on resume."""
    tmp = path + ".tmp.png"
    img.save(tmp)
    os.replace(tmp, path)


_pre_resize = _pre_deleter = None

def _pre_init():
    global _pre_resize, _pre_deleter
    import cv2
    cv2.setNumThreads(0)
    _pre_resize = transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR, antialias=True)
    _pre_deleter = LineCorruption(**CORRUPTION_KWARGS)


def _pre_todo(stem):
    """Variant indices still missing for this image."""
    return [k for k in range(N_VARIANTS)
            if not os.path.exists(os.path.join(pre_corr, f"{stem}_v{k}.png"))]


def _pre_job(args):
    idx, fname = args
    stem = os.path.splitext(fname)[0]
    clean_path = os.path.join(pre_clean, stem + ".png")
    todo = _pre_todo(stem)
    if not todo and os.path.exists(clean_path):
        return 0, 0

    img = Image.open(os.path.join(clean_src, fname)).convert("L")
    img = _pre_resize(img)                       # same resize as training / test
    if not os.path.exists(clean_path):
        _save_atomic(img, clean_path)

    empty = 0
    for k in todo:
        random.seed(PRE_SEED + idx * 1000 + k)   # reproducible per (image, variant)
        cor = _pre_deleter(img)
        if cor is img:                           # nothing erased
            empty += 1
            cor = img.copy()
        _save_atomic(cor, os.path.join(pre_corr, f"{stem}_v{k}.png"))
    return len(todo), empty


# idx comes from the FULL sorted list, so seeds stay the same when resuming
all_jobs = list(enumerate(sorted(train_files)))

# Only send images that still need work -> the progress bar and ETA reflect
# the real remaining work, even when resuming after a disconnect
jobs = [(i, f) for i, f in all_jobs
        if _pre_todo(os.path.splitext(f)[0])
        or not os.path.exists(os.path.join(pre_clean, os.path.splitext(f)[0] + ".png"))]
n_missing = sum(len(_pre_todo(os.path.splitext(f)[0])) for _, f in jobs)

print(f"{len(all_jobs)} training images x {N_VARIANTS} variants on {N_PROC} processes")
print(f"already done: {len(all_jobs) - len(jobs)} images | "
      f"to do: {len(jobs)} images ({n_missing} pairs)")

t0, made, empty = time.perf_counter(), 0, 0
state = random.getstate()
if jobs:
    with get_context("fork").Pool(N_PROC, initializer=_pre_init) as pool:
        bar = tqdm(pool.imap_unordered(_pre_job, jobs, chunksize=1),
                   total=len(jobs), desc="train pairs", unit="img")
        for m, e in bar:
            made, empty = made + m, empty + e
            bar.set_postfix(new_pairs=made, empty=empty)
random.setstate(state)
dt = time.perf_counter() - t0

with open(pre_manifest, "w") as fh:
    json.dump(dict(settings=pre_settings, n_variants=N_VARIANTS, complete=True), fh, indent=2)
print(f"Done in {dt/60:.1f} min: {made} new pairs, {empty} with nothing erased -> {pre_root}")
if jobs:
    print(f"({dt / len(jobs):.2f} s per image with {N_PROC} processes, "
          f"{dt / max(made, 1) * N_PROC:.2f} s per pair per process)")


# ***SAVE TO DISK***

**Dataset zip layout** (`/content/inkrepair_dataset.zip`, all images 896x896 grayscale PNG):

- `split/split.json`: split manifest (seed, image size, test fraction, corruption settings, corruption report, and the train / test / sample file lists)
- `split/clean_test/<name>.png`: ~10% of the dataset held out from training, clean and resized
- `split/corrupted_test/<name>.png`: the frozen corrupted version of each test image
- `split/precorrupted/manifest.json`: settings used to generate the training pairs, number of variants and completion flag
- `split/precorrupted/clean/<name>.png`: clean training images (~90% of the dataset), resized
- `split/precorrupted/corrupted/<name>_v0.png ... <name>_v5.png`: 6 corrupted versions of each training image; the loader picks one at random each time


In [ ]:
!du -sh /content/split/precorrupted /content/split/clean_test /content/split/corrupted_test

In [ ]:
!cd /content && zip -r -0 -q inkrepair_dataset.zip split -x "split/clean_training/*" "*.tmp.png"
!ls -lh /content/inkrepair_dataset.zip

In [ ]:
from google.colab import files
files.download("/content/inkrepair_dataset.zip")